In [1]:
!pip install tensorflow numba pandas numpy

In [4]:
import tensorflow as tf
import numpy as np
from numba import cuda
import time

# =========================
# Load MNIST Dataset
# =========================

(x_train, y_train), (_, _) = tf.keras.datasets.mnist.load_data()

pixels = x_train.flatten().astype(np.uint8)

print("Total Pixels:", len(pixels))

# =========================
# Global Memory Histogram
# =========================

@cuda.jit
def histogram_global(img, hist):

    idx = cuda.grid(1)

    if idx < img.size:
        pixel = img[idx]
        cuda.atomic.add(hist, pixel, 1)

# =========================
# Shared Memory Histogram
# =========================

@cuda.jit
def histogram_shared(img, hist):

    shared_hist = cuda.shared.array(
        shape=256,
        dtype=np.int32
    )

    tid = cuda.threadIdx.x
    idx = cuda.grid(1)

    # Initialize shared histogram
    if tid < 256:
        shared_hist[tid] = 0

    cuda.syncthreads()

    # Local histogram accumulation
    if idx < img.size:
        pixel = img[idx]
        cuda.atomic.add(shared_hist, pixel, 1)

    cuda.syncthreads()

    # Merge shared histogram into global histogram
    if tid < 256:
        cuda.atomic.add(
            hist,
            tid,
            shared_hist[tid]
        )

# =========================
# GPU Setup
# =========================

threads_per_block = 256
blocks_per_grid = (
    pixels.size + threads_per_block - 1
) // threads_per_block

d_pixels = cuda.to_device(pixels)

# =========================
# Global Memory Version
# =========================

hist_global = np.zeros(256, dtype=np.int32)
d_hist_global = cuda.to_device(hist_global)

start = time.time()

histogram_global[
    blocks_per_grid,
    threads_per_block
](
    d_pixels,
    d_hist_global
)

cuda.synchronize()

global_time = time.time() - start

hist_global = d_hist_global.copy_to_host()

# =========================
# Shared Memory Version
# =========================

hist_shared = np.zeros(256, dtype=np.int32)
d_hist_shared = cuda.to_device(hist_shared)

start = time.time()

histogram_shared[
    blocks_per_grid,
    threads_per_block
](
    d_pixels,
    d_hist_shared
)

cuda.synchronize()

shared_time = time.time() - start

hist_shared = d_hist_shared.copy_to_host()

# =========================
# Results
# =========================

print("\nGlobal Memory Histogram:")
print(hist_global)

print("\nShared Memory Histogram:")
print(hist_shared)

print("\nResults Comparison")
print("----------------------")
print(f"Global Memory Time : {global_time:.6f} sec")
print(f"Shared Memory Time : {shared_time:.6f} sec")

print("\nHistograms Equal:",
      np.array_equal(hist_global, hist_shared))

Total Pixels: 47040000

Global Memory Histogram:
[38045844    22896    33653    36040    38267    39148    37692    38856
    30878    38234    35282    36020    30139    40100    26939    28869
    29115    27551    26849    34431    29955    35496    26750    22910
    25950    29995    24260    24025    25434    37160    22913    26205
    28890    15556    19906    21516    22128    24760    25922    18250
    20675    27023    22349    21227    19030    21122    17326    24237
    20083    17919    23964    25003    14588    19230    18195    18068
    23511    31905    14330    18140    18144    18133    19805    23909
    46754    16050    17514    15914    16302    16742    19288    18444
    17313    19307    13816    15875    17877    13535    17569    18085
    15872    16527    21112    15514    27088    25496    25837    12645
    15796    17628    12695    17876    18525    17225    16655    16244
    17902    14246    16820    17710    15217    14210    21721    14854
  